# Tutorial 8: Single-Cell Foundation Model Embeddings

Cell embeddings are now demonstrated through `BioEmbedder.embed(...)` with
`entity_type="cell"`. The returned AnnData keeps cell-level matrices in
`.obsm` and standardized provenance in `.uns["embeddings"]`.


In [ ]:
from embpy import BioEmbedder
import anndata as ad
import numpy as np
import pandas as pd

RUN_EMBEDDING = False
embedder = BioEmbedder(device="auto", organism="human")

rng = np.random.default_rng(7)
adata = ad.AnnData(
    X=np.abs(rng.normal(size=(40, 300))).astype(np.float32),
    obs=pd.DataFrame({"batch": ["a"] * 20 + ["b"] * 20}),
)
adata.obs_names = [f"cell_{i}" for i in range(adata.n_obs)]
adata.var_names = [f"Gene_{i}" for i in range(adata.n_vars)]
adata


## 1. PCA cell embeddings


In [ ]:
if RUN_EMBEDDING:
    cell_adata = embedder.embed(
        adata,
        entity_type="cell",
        model="pca",
        output="anndata",
        preprocessing="standard",
        n_pca_components=16,
        n_top_genes=100,
        key="X_pca",
    )
    print(cell_adata.obsm["X_pca"].shape)
    print(cell_adata.uns["embeddings"]["X_pca"])


## 2. Foundation models use the same surface


In [ ]:
if RUN_EMBEDDING:
    scgpt_adata = embedder.embed(
        adata,
        entity_type="cell",
        model="scgpt",
        output="anndata",
        preprocessing="standard",
        batch_size=16,
        key="X_scgpt",
    )
    print(scgpt_adata.obsm["X_scgpt"].shape)


## 3. Multiple cell embeddings in one call


In [ ]:
if RUN_EMBEDDING:
    multi_cell = embedder.embed(
        adata,
        entity_type="cell",
        model=["pca", "scgpt"],
        output="anndata",
        preprocessing="standard",
        n_pca_components=16,
        n_top_genes=100,
        batch_size=16,
    )
    print(list(multi_cell.obsm.keys()))
    print(list(multi_cell.uns["embeddings"].keys()))


## 4. Payload output for audit trails


In [ ]:
if RUN_EMBEDDING:
    payload = embedder.embed(
        adata,
        entity_type="cell",
        model="pca",
        output="payload",
        preprocessing="standard",
        n_pca_components=16,
        n_top_genes=100,
        key="X_pca",
    )
    print(payload["entity_type"], payload["n_entities"], payload["n_dims"])
